In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import parselmouth
from tqdm import tqdm
# Removed: from itertools import islice

AUDIO_FOLDER = "Cleaned_Audio"
OUTPUT_CSV = "features_full.csv" # Changed output filename
TRANSCRIPT_FOLDER = "cleaned_transcripts"
MERGED_LABELS_PATH = "merged_labels.csv"

def compute_articulation_speed(transcript_path, speech_duration):

    df = pd.read_csv(transcript_path)

    # Combine all participant text
    full_text = " ".join(df["value"].astype(str))

    # Count words
    word_count = len(full_text.split())

    # Avoid division by zero
    if speech_duration > 0:
        articulation_speed = word_count / speech_duration
    else:
        articulation_speed = 0

    return articulation_speed, word_count

def extract_audio_features(file_path):

    y, sr = librosa.load(file_path, sr=16000)
    features = {}

    total_duration = librosa.get_duration(y=y, sr=sr)

    # -----------------------
    # Silence / speech
    # -----------------------
    intervals = librosa.effects.split(y, top_db=30)

    speech_duration = sum([(end - start)/sr for start, end in intervals])
    silence_duration = total_duration - speech_duration

    features["speech_duration"] = speech_duration
    features["silence_duration"] = silence_duration
    features["speech_to_silence_ratio"] = speech_duration / (silence_duration + 1e-6)
    features["silence_ratio"] = silence_duration / total_duration

    # -----------------------
    # RMS / Energy
    # -----------------------
    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["energy_mean"] = features["rms_mean"]
    features["energy_std"] = features["rms_std"]

    # -----------------------
    # ZCR
    # -----------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # -----------------------
    # MFCC
    # -----------------------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # -----------------------
    # Spectral features
    # -----------------------
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]

    features["spectral_centroid_mean"] = np.mean(centroid)
    features["spectral_centroid_std"] = np.std(centroid)

    features["spectral_bandwidth_mean"] = np.mean(bandwidth)
    features["spectral_bandwidth_std"] = np.std(bandwidth)

    features["spectral_rolloff_mean"] = np.mean(rolloff)
    features["spectral_rolloff_std"] = np.std(rolloff)

    # -----------------------
    # Pitch
    # -----------------------
    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7')
    )

    f0 = f0[~np.isnan(f0)]

    if len(f0) > 0:
        features["pitch_mean"] = np.mean(f0)
        features["pitch_std"] = np.std(f0)
    else:
        features["pitch_mean"] = 0
        features["pitch_std"] = 0

    # -----------------------
    # Speaking rate (approx)
    # -----------------------
    features["speaking_rate"] = len(intervals) / total_duration

    # -----------------------
    # Pause duration
    # -----------------------
    pause_durations = []
    prev_end = 0

    for start, end in intervals:
        pause = (start/sr) - (prev_end/sr)
        if pause > 0:
            pause_durations.append(pause)
        prev_end = end

    features["pause_duration"] = np.mean(pause_durations) if pause_durations else 0

    # -----------------------
    # Jitter & Shimmer
    # -----------------------
    snd = parselmouth.Sound(file_path)
    point_process = parselmouth.praat.call(
        snd, "To PointProcess (periodic, cc)", 75, 500
    )

    jitter = parselmouth.praat.call(
        point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3
    )

    shimmer = parselmouth.praat.call(
        [snd, point_process], "Get shimmer (local)",
        0, 0, 0.0001, 0.02, 1.3, 1.6
    )

    features["jitter"] = jitter
    features["shimmer"] = shimmer

    return features


# -----------------------
# MAIN LOOP
# -----------------------

all_features = []

# Load merged labels and get allowed participant IDs
try:
    merged_labels_df = pd.read_csv(MERGED_LABELS_PATH)
    allowed_participants = set(merged_labels_df["Participant_ID"].astype(str))
    print(f"Loaded {len(allowed_participants)} participants from merged labels.")
except FileNotFoundError:
    print(f"Error: Merged labels file not found at {MERGED_LABELS_PATH}")
    allowed_participants = set() # Process no files if labels not found

audio_files = [f for f in os.listdir(AUDIO_FOLDER) if f.endswith(".wav")]

print("Total audio files in folder:", len(audio_files))

# Process all participants
for file in tqdm(audio_files):

    file_path = os.path.join(AUDIO_FOLDER, file)

    participant_id = file.split("_")[0]

    # Skip if participant is not in the allowed list
    if participant_id not in allowed_participants:
        # print(f"Skipping file {file} as participant {participant_id} not in merged labels.")
        continue

    try:
        features = extract_audio_features(file_path)
        # Path to transcript
        transcript_path = os.path.join(
            TRANSCRIPT_FOLDER,
            f"{participant_id}_TRANSCRIPT_cleaned.csv"
        )

        # Compute articulation speed
        articulation_speed, word_count = compute_articulation_speed(
            transcript_path,
            features["speech_duration"]
        )

        # Add to features
        features["articulation_speed"] = articulation_speed
        features["word_count"] = word_count # User rejected removal of word_count, so keep it
        features["participant_id"] = participant_id

        # Add 'label' column from merged_labels_df
        label = merged_labels_df[merged_labels_df["Participant_ID"].astype(str) == participant_id]["PHQ8_Binary"].values
        if len(label) > 0:
            features["label"] = label[0]
        else:
            features["label"] = np.nan # Or any other default value for missing labels

        all_features.append(features)

    except Exception as e:
        print("Error in file:", file, e)

# -----------------------
# SAVE CSV
# -----------------------

df = pd.DataFrame(all_features)

# Reorder columns to have 'participant_id' as the first column and 'label' as the last
if 'participant_id' in df.columns and 'label' in df.columns:
    cols = df.columns.tolist()
    cols.remove('participant_id')
    cols.remove('label')
    df = df[['participant_id'] + cols + ['label']]
elif 'participant_id' in df.columns:
    cols = df.columns.tolist()
    cols.remove('participant_id')
    df = df[['participant_id'] + cols]

df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)


Loaded 142 participants from merged labels.
Total audio files in folder: 188


100%|██████████| 188/188 [5:37:26<00:00, 107.70s/it]  


Saved: features_full.csv


In [3]:
import pandas as pd

# Load files
merged_labels = pd.read_csv("merged_labels.csv")
features = pd.read_csv("features_full.csv")

# Extract participant IDs
merged_ids = merged_labels["Participant_ID"].astype(str)
feature_ids = features["participant_id"].astype(str)

# Print them
print("Merged Labels Participant IDs:")
print(merged_ids.tolist())

print("\nFeature File Participant IDs:")
print(feature_ids.tolist())

Merged Labels Participant IDs:
['303', '304', '305', '310', '312', '313', '315', '316', '317', '318', '319', '320', '321', '322', '324', '325', '326', '327', '328', '330', '333', '336', '338', '339', '340', '341', '343', '344', '345', '347', '348', '350', '351', '352', '353', '355', '356', '357', '358', '360', '362', '363', '364', '366', '368', '369', '370', '371', '372', '374', '375', '376', '379', '380', '383', '385', '386', '391', '392', '393', '397', '400', '401', '402', '409', '412', '414', '415', '416', '419', '423', '425', '426', '427', '428', '429', '430', '433', '434', '437', '441', '443', '444', '445', '446', '447', '448', '449', '454', '455', '456', '457', '459', '463', '464', '468', '471', '473', '474', '475', '478', '479', '485', '486', '487', '488', '491', '302', '307', '331', '335', '346', '367', '377', '381', '382', '388', '389', '390', '395', '403', '404', '406', '413', '417', '418', '420', '422', '436', '439', '440', '451', '458', '472', '476', '477', '482', '483', '4